| [![back](../../media/navigation/back.png)](../../exercises/cells-detection-segmentation/ex3.html) | [![home](../../media/navigation/home.png)](../../index.html) |
| :---: | :---: |
| Ex 5.4: Use CellPose in QuPath | • |

# 5. Cells detection

## 5.5 Use InstanSeg in QuPath

InstanSeg is the last deep-learning architecture that we will cover. It is also maid for bloboid objects, but the models shipped by default with the plugin are limited to nuclei (IHC RGB, IHC deconvolued and fluo).

The pixel size is mandatory for this architecture because rescaling is performed automatically.

**Note**

For both upcoming exercises (5.5.1 and 5.5.2), keep your projects clean and save them often, we will reuse one of them (the one you prefer) into the next chapter.

### 5.5.1 Count nuclei in lung biopsy

#### Goals:

- We want to count the number of nuclei in the same lung biopsies as the one we used in [Ex 4.2.1: Compactness of lung per image](../../exercises/tissue-classification-segmentation/ex2.html#4.2.1-Compactness-of-lung-per-image).
- We want to launch the segmentation only in the areas of lung found by the pixel classifier.
- To do so, we will:
    - start from the project in which the lungs are already segmented
    - find a way to isolate lungs annotations
    - run InstanSeg inside

#### Required data:

| **Folder** | Description | Location | License |
| --- | --- | --- | --- |
| Histology (PAS-HE-IHC) | H-DAB images of kidneys into which glomerulus are visible and lungs into which alveolus are present | [DOI: 10.1038/s41467-023-40291-0](https://www.kaggle.com/datasets/yashvrdnjain/histology-pas-he-ihc-images-ftu-segmentation) | MIT |

### A. Locate the lungs tissue

- If you started from the project that you created for [Ex 4.2.1: Compactness of lung per image](../../exercises/tissue-classification-segmentation/ex2.html#4.2.1-Compactness-of-lung-per-image), you should already have the polygons around each areas of lung.
- If you chose to start over from a new project, you can visit the other project's folder to copy and paste the pixel classifier into the current project.
- We are now left with a project into which we have two polygons on each image:
    - one using the `Hull` classification
    - the other with the `Lung` classification
- So far, we launched everything we have ever made in a full image annotation, but we can work in arbitrary annotations as well! We just need to find a way to make them active on the flight.
- In this situation, it is very easy thanks to the `Lung` class.
- If you do a ![Kenney-right click](../../media/inputs-icons/mouse_right.svg) right-click on the name of the `Lung` class into the class list, you will be given the option to "Select objects by classification". You can use it right away.
- Every bit of lung tissue should now be active.

### B. Run InstanSeg

- InstanSeg resizes automatically images to match the required object size (in pixels) due to that, you must provide the pixel size for your images. To do so, go to the "Image" tab (alongside the "Annotations", "Workflow", ... tabs). 
- The pixel width and height should be unset, but by double-clicking on either fields, you can set them manually to 0.6535µm.
- If you installed InstanSeg as described in the requirements, you should find in the top-menu bar, in the "Extensions" menu, an "InstanSeg" entry into which you have only one feature called "Run InstanSeg". Launch it to display segmentation settings.
- The first time that you will launch InstanSeg, you will have to provide a path for a folder into which the models will be saved. To do that:
    - Using the ![QP path](../../media/qp-icons/folder.png) "choose path" button, either choose to use the default path (recommended) or use a custom path.
    - Then, in the dropdown menu, select the "brightfield-nuclei" model: it's the one to use for RGB images.
    - Eventually, click on the ![QP dl](../../media/qp-icons/download.png) "download" button to launch the downloading of the model.
- Once it is over, it is time to use our newly downloaded model.
- In the "Input channels", make sure that the R, G and B channels are being used.
- In the "Output type", use some detections.
- For the next chapter sake, make sure that the "Make measurements" option is activated.
- If everything is set, feel free to run the segmentation.

![cells lung](../../media/cell-detection/cells-lung.png)

### C. Run for the project

- In the "Workflow" tab, convert your commands history into a script.
- In here, three things are going to be interesting for us:
    - The command to set the pixel size
    - The selection of objects by `Lung` classification
    - The instanciation of the InstanSeg object (on several lines)
- You should be left with a script looking like this:

```groovy
setPixelSizeMicrons(0.653500, 0.653500)
selectObjectsByClassification("Lung")
qupath.ext.instanseg.core.InstanSeg.builder()
    .modelPath("/home/clement/QuPath/v0.6/instanseg/downloaded/brightfield_nuclei-0.1.1")
    .device("gpu0")
    .inputChannels([ColorTransforms.createChannelExtractor("Red"), ColorTransforms.createChannelExtractor("Green"), ColorTransforms.createChannelExtractor("Blue")])
    .outputChannels()
    .tileDims(512)
    .interTilePadding(32)
    .nThreads(4)
    .makeMeasurements(true)
    .randomColors(false)
    .outputType("Detections")
    .build()
    .detectObjects()
```

- You can run it for all the images, except the one you already processed.